<a href="https://colab.research.google.com/github/MAhmad25/Emotion_Detection_using_NLP/blob/master/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [70]:
import pandas as pd

In [71]:
df=pd.read_csv("sample_data/train.txt",sep=";",header=None,names=["Text","Emotions"])

In [72]:
df.head()

,Text,Emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [73]:
df.isnull().sum()

,0
Text,0
Emotions,0


In [74]:
df=df.drop_duplicates()

In [75]:
unique_emotion=df["Emotions"].unique()

In [76]:
unique_emotion

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [77]:
# Mapping Target Value into Numbers
unique={}
i=0
for emo in unique_emotion:
        unique[emo]=i
        i+=1
df["Emotions"]=df["Emotions"].map(unique).astype(int)
df

,Text,Emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


# Text Processing

<h3>Convert to Lowercase</h3>

In [78]:
def toLowerCase(text:str):
        return text.lower()
df["Text"]=df["Text"].apply(toLowerCase)

# Remove URLs

In [79]:
import re

def remove_urls(text: str) -> str:
    pattern = r'https?://\S+|www\.\S+'
    return re.sub(pattern, '', text)

df["Text"]=df["Text"].apply(remove_urls)

# Remove Digits

In [80]:
def remove_digits(text: str):
    return re.sub(r'\d+', '', text)

df["Text"]=df["Text"].apply(remove_digits)

# Remove Emojies

In [81]:

def remove_emojis(text: str) -> str:
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002700-\U000027BF"
        "\U0001F900-\U0001F9FF"
        "\U00002600-\U000026FF"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub('', text)

df["Text"]=df["Text"].apply(remove_emojis)

# Remove Punctuations

In [82]:
import string

def remove_punctuation(text: str) -> str:
    return text.translate(str.maketrans('', '', string.punctuation))

df["Text"]=df["Text"].apply(remove_punctuation)

In [83]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [84]:
def remove_stop_word(text:str)->str:
        tokens_word=word_tokenize(text)
        filtered_word=[
                word for word in tokens_word
                if word.lower() not in stop_words
        ]
        return " ".join(filtered_word)

remove_stop_word('i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake')

'go feeling hopeless damned hopeful around someone cares awake'

<h3>Remove Stop words because they don't effect in NLP</h3>

In [85]:
df["cleaned_text"]=df["Text"].apply(remove_stop_word)
df.tail()

,Text,Emotions,cleaned_text
15995,i just had a very brief time in the beanbag an...,0,brief time beanbag said anna feel like beaten
15996,i am now turning and i feel pathetic that i am...,0,turning feel pathetic still waiting tables sub...
15997,i feel strong and good overall,5,feel strong good overall
15998,i feel like this was such a rude comment and i...,1,feel like rude comment im glad
15999,i know a lot but i feel so stupid because i ca...,0,know lot feel stupid portray


<h3>Using Bag of words instead of One-hot encoding to convert documents into vectors</h3>

In [86]:
# Ignore this code as i was testing how it works TESTING
from sklearn.feature_extraction.text import CountVectorizer
vect=CountVectorizer(ngram_range=(2,2))
documents = [
    "I like Python",
    "AI is useful",
    "Machine learning is interesting"
]
model=vect.fit_transform(documents)

In [87]:
print(vect.get_feature_names_out(documents))
print(model.toarray()) # type: ignore

['ai is' 'is interesting' 'is useful' 'learning is' 'like python'
 'machine learning']
[[0 0 0 0 1 0]
 [1 0 1 0 0 0]
 [0 1 0 1 0 1]]


In [88]:
from sklearn.metrics import recall_score,precision_score,accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
BOW=CountVectorizer(ngram_range=(2,2))
TF_IDF=TfidfVectorizer(ngram_range=(2,2))
X_train, X_test, y_train, y_test = train_test_split(df["cleaned_text"], df["Emotions"], test_size=0.20, random_state=42)

<h3>Using Bag of words</h3>

In [89]:
X_train_BOW=BOW.fit_transform(X_train)
X_test_BOW=BOW.transform(X_test)

models={
        "Multinomial NaiveByes":MultinomialNB(),
        "Logistic Regression":LogisticRegression(max_iter=1000),
        "Support Vector Machine":SVC(kernel="sigmoid"),
        "XGBOOST": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, use_label_encoder=True, eval_metric='mlogloss', random_state=42)
}

results=[]
for name,model in models.items():
        model.fit(X_train_BOW,y_train)
        y_predict=model.predict(X_test_BOW)
        results.append({
                "Model Name: ":name,
                "Accuracy: ":accuracy_score(y_test,y_predict),
                "Precision: ":precision_score(y_test,y_predict,average="weighted"),
                "Recall: ":recall_score(y_test,y_predict,average="weighted")
        })

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:47:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [90]:
pd.DataFrame(data=results)

,Model Name:,Accuracy:,Precision:,Recall:
0,Multinomial NaiveByes,0.661250,0.721250,0.661250
1,Logistic Regression,0.683750,0.738105,0.683750
2,Support Vector Machine,0.646563,0.750161,0.646563
3,XGBOOST,0.503437,0.736483,0.503437


<h3>Using TF-IDF</h3>

In [91]:
X_train_TF_IDF=TF_IDF.fit_transform(X_train)
X_test_TF_IDF=TF_IDF.transform(X_test)

models={
        "Multinomial NaiveByes":MultinomialNB(),
        "Logistic Regression":LogisticRegression(max_iter=1000),
        "Support Vector Machine":SVC(kernel="sigmoid"),
        "XGBOOST": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, use_label_encoder=True, eval_metric='mlogloss', random_state=42)

}

results=[]
for name,model in models.items():
        model.fit(X_train_BOW,y_train)
        y_predict=model.predict(X_test_BOW)
        results.append({
                "Model Name":name,
                "Accuracy":accuracy_score(y_test,y_predict),
                "Precision":precision_score(y_test,y_predict,average="weighted"),
                "Recall":recall_score(y_test,y_predict,average="weighted")
        })

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:48:40] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [92]:
pd.DataFrame(results)

,Model Name,Accuracy,Precision,Recall
0,Multinomial NaiveByes,0.661250,0.721250,0.661250
1,Logistic Regression,0.683750,0.738105,0.683750
2,Support Vector Machine,0.646563,0.750161,0.646563
3,XGBOOST,0.503437,0.736483,0.503437


### Hyperparameter Tuning with Randomized Search Cross-Validation

In [93]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

In [94]:
# Define parameter distributions for each model
param_dist_mnb = {
    'alpha': uniform(0.01, 1.0)
}

param_dist_lr = {
    'C': uniform(0.1, 10.0),
    'solver': ['liblinear', 'saga']
}

param_dist_svc = {
    'C': uniform(0.1, 10.0),
    'gamma': ['scale', 'auto', uniform(0.001, 1.0)]
}

param_dist_xgb = {
    'n_estimators': randint(50, 200),
    'learning_rate': uniform(0.01, 0.3),
    'max_depth': randint(3, 10),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4)
}

param_distributions = {
    "Multinomial NaiveByes": param_dist_mnb,
    "Logistic Regression": param_dist_lr,
    "Support Vector Machine": param_dist_svc,
    "XGBOOST": param_dist_xgb
}

In [ ]:
tuned_results = []

for name, model in models.items():
    print(f"Tuning {name}...")
    param_dist = param_distributions[name]

    # Use TF-IDF transformed data for tuning, assuming it might perform better
    # or that the user wants to tune for this representation
    X_train_data = X_train_TF_IDF
    X_test_data = X_test_TF_IDF

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=10,  # Number of parameter settings that are sampled
        cv=3,       # Number of folds in cross-validation
        scoring='accuracy',
        random_state=42,
        n_jobs=-1   # Use all available cores
    )

    random_search.fit(X_train_data, y_train)

    best_model = random_search.best_estimator_
    y_predict = best_model.predict(X_test_data)

    tuned_results.append({
        "Model Name": f"{name} (Tuned)",
        "Best Parameters": random_search.best_params_,
        "Accuracy": accuracy_score(y_test, y_predict),
        "Precision": precision_score(y_test, y_predict, average="weighted"),
        "Recall": recall_score(y_test, y_predict, average="weighted")
    })
    print(f"Finished tuning {name}.")

Tuning Multinomial NaiveByes...
Finished tuning Multinomial NaiveByes.
Tuning Logistic Regression...
Finished tuning Logistic Regression.
Tuning Support Vector Machine...


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
6 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 436, in _validate_params
    validate_parameter_constraints(
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/_pa

Finished tuning Support Vector Machine.
Tuning XGBOOST...


In [ ]:
import pandas as pd
pd.DataFrame(tuned_results)